In [ ]:
import triton
import triton.language as tl
import torch
import math

device = 'cuda' if  torch.cuda.is_available() else 'cpu'
one_dim_len = 1000000
x = torch.arange(one_dim_len, dtype=torch.float32).to(device)
y = torch.arange(one_dim_len, dtype=torch.float32).to(device)

@triton.jit
def vector_addition_kernel(
    a_p,
    b_p,
    c_p,
    num,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  block_id = pid*BLOCK_SIZE
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets<num

  a = tl.load(a_p + offsets, mask = mask)
  b = tl.load(b_p + offsets, mask = mask)

  c = a+b
  tl.store(c_p + offsets, c, mask=mask)

def vector_add(a : torch.Tensor, b: torch.Tensor) -> torch.Tensor:
  assert device == 'cuda'
  assert a.shape == b.shape
  assert a.is_cuda
  assert b.is_cuda
  assert a.is_contiguous()
  assert b.is_contiguous()
  num = a.numel()
  c = torch.empty_like(a)

  grid = lambda meta: (triton.cdiv(num, meta['BLOCK_SIZE']),)

  vector_addition_kernel[grid](
      a,
      b,
      c,
      num,
      BLOCK_SIZE = 256
  )
  return c

vector_add(x,y)

tensor([0.0000e+00, 2.0000e+00, 4.0000e+00,  ..., 2.0000e+06, 2.0000e+06,
        2.0000e+06], device='cuda:0')

In [ ]:
@triton.jit
def silu_kernel(
    x_p,
    out_p,
    num,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  block_id = pid*BLOCK_SIZE
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets<num

  x = tl.load(x_p + offsets, mask=mask)
  t = tl.sigmoid(x)
  x = x*t
  tl.store(out_p+offsets, x, mask=mask)

def silu(x : torch.Tensor):
  assert x.is_cuda
  assert x.is_contiguous()
  out = torch.empty_like(x)
  num = x.numel()

  grid = lambda meta : (triton.cdiv(num, meta['BLOCK_SIZE']),)

  silu_kernel[grid](
      x,
      out,
      num,
      BLOCK_SIZE = 1024
  )
  return out

silu(x)

tensor([0.0000e+00, 7.3106e-01, 1.7616e+00,  ..., 1.0000e+06, 1.0000e+06,
        1.0000e+06], device='cuda:0')

In [ ]:
@triton.jit
def gelu_kernel(
    x_p,
    out_p,
    num,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  block_id = pid*BLOCK_SIZE
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets<num

  xo = tl.load(x_p + offsets, mask=mask)
  x = xo.to(tl.float32)
  x2 = x*x*x*0.044715 + x
  x2 = x2*0.7978845608
  tanh_val = 2.0 * tl.sigmoid(2.0 * x2) - 1.0
  x2 = 0.5 * x * (1.0 + tanh_val)
  out = x2.to(xo.dtype)
  tl.store(out_p + offsets, x2, mask = mask)


def gelu(x : torch.Tensor):
  assert x.is_cuda
  assert x.is_contiguous()
  out = torch.empty_like(x)
  num = x.numel()

  grid = lambda meta : (triton.cdiv(num, meta['BLOCK_SIZE']),)

  gelu_kernel[grid](
      x,
      out,
      num,
      BLOCK_SIZE = 1024
  )
  return out

gelu(x)


tensor([0.0000e+00, 8.4119e-01, 1.9546e+00,  ..., 1.0000e+06, 1.0000e+06,
        1.0000e+06], device='cuda:0')

In [ ]:
@triton.jit
def matrix_add_kernel(
    x_ptr,
    y_ptr,
    out_ptr,
    M,
    N,
    stride_m, stride_n,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr
):
  pid_m = tl.program_id(0)
  pid_n= tl.program_id(1)

  block_idm = pid_m*BLOCK_M
  block_idn = pid_n*BLOCK_N

  offsets_m = block_idm + tl.arange(0, BLOCK_M)
  offsets_n = block_idn + tl.arange(0, BLOCK_N)

  offset_matrix = offsets_m[:,None]*stride_m + offsets_n[None,:]*stride_n

  x_ptr = x_ptr + offset_matrix
  y_ptr = y_ptr + offset_matrix
  out_ptr = out_ptr + offset_matrix

  mask = (offsets_m[:,None] < M) & (offsets_n[None,:] <N)

  x = tl.load(x_ptr, mask=mask, other=0.0)
  y = tl.load(y_ptr, mask=mask, other=0.0)

  out = x+y

  tl.store(out_ptr, out, mask=mask)

def matrix_add(x : torch.Tensor, y : torch.Tensor):
  assert x.is_contiguous()
  assert y.is_contiguous()
  assert x.is_cuda
  assert y.is_cuda
  assert x.shape == y.shape

  M, N = x.shape
  out = torch.empty_like(x)

  BLOCK_M = 32
  BLOCK_N = 32

  grid = lambda meta : (triton.cdiv(M , meta['BLOCK_M']), triton.cdiv(N, meta['BLOCK_N']))

  matrix_add_kernel[grid](
      x,
      y,
      out,
      M,
      N,
      x.stride(0), x.stride(1),
      BLOCK_M=BLOCK_M,
      BLOCK_N=BLOCK_N,
  )
  return out

x2d = torch.arange(10000, dtype=torch.float32).view(100,100).to(device)
y2d = torch.arange(10000, dtype=torch.float32).view(100,100).to(device)
print(matrix_add(x2d,y2d))

tensor([[0.0000e+00, 2.0000e+00, 4.0000e+00,  ..., 1.9400e+02, 1.9600e+02,
         1.9800e+02],
        [2.0000e+02, 2.0200e+02, 2.0400e+02,  ..., 3.9400e+02, 3.9600e+02,
         3.9800e+02],
        [4.0000e+02, 4.0200e+02, 4.0400e+02,  ..., 5.9400e+02, 5.9600e+02,
         5.9800e+02],
        ...,
        [1.9400e+04, 1.9402e+04, 1.9404e+04,  ..., 1.9594e+04, 1.9596e+04,
         1.9598e+04],
        [1.9600e+04, 1.9602e+04, 1.9604e+04,  ..., 1.9794e+04, 1.9796e+04,
         1.9798e+04],
        [1.9800e+04, 1.9802e+04, 1.9804e+04,  ..., 1.9994e+04, 1.9996e+04,
         1.9998e+04]], device='cuda:0')


In [ ]:
@triton.jit
def oned_softmax_kernel(
    x_ptr,
    out_ptr,
    M, N,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  # assert that N is less than block size before using
  block_id = pid*N
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets < (pid+1)*N

  x = tl.load(x_ptr+offsets, mask=mask, other = -float('inf'))

  x_max = tl.max(x, axis = 0)
  x = x - x_max

  x_numerator = tl.exp(x)
  x_numerator_safe = tl.where(mask, x_numerator, 0.0)
  x_denominator = tl.sum(x_numerator_safe, axis = 0)

  out = x_numerator / x_denominator

  tl.store(out_ptr+offsets, out, mask=mask)

def oned_softmax(x: torch.Tensor, M, N):
  assert x.is_cuda
  assert x.is_contiguous()
  #print(x.shape)
  assert x.shape[0] == M*N
  assert N < 1024

  num = x.numel()
  out = torch.empty_like(x)

  grid = lambda meta: (M,)

  oned_softmax_kernel[grid](
      x,
      out,
      M, N,
      BLOCK_SIZE = 1024
  )

  return out

oned_softmax(x,1000,1000)


tensor([0.0000, 0.0000, 0.0000,  ..., 0.0855, 0.2325, 0.6321], device='cuda:0')

In [ ]:
@triton.jit
def matrix_transpose_kernel(
    x_ptr,
    out_ptr,
    M, N,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr
):
  pid_m = tl.program_id(0)
  pid_n = tl.program_id(1)

  block_idm = pid_m*BLOCK_M
  block_idn = pid_n*BLOCK_N

  offsets_m = block_idm + tl.arange(0, BLOCK_M)
  offsets_n = block_idn + tl.arange(0, BLOCK_N)

  offset_matrix_x = offsets_m[:,None]*N + offsets_n[None,:]
  offset_matrix_out = offsets_n[:,None]*M + offsets_m[None,:]

  x_ptr = x_ptr + offset_matrix_x
  out_ptr = out_ptr + offset_matrix_out

  x_mask = (offsets_m[:,None]<M) &  (offsets_n[None,:]<N)
  out_mask = (offsets_n[:,None]<N) &  (offsets_m[None,:]<M)

  x = tl.load(x_ptr, mask=x_mask, other=0.0)

  out = tl.trans(x)
  tl.store(out_ptr, out, out_mask)

def matrix_transpose(x : torch.Tensor):
  assert x.is_contiguous()
  assert x.is_cuda

  out = torch.empty_like(x).T.contiguous()
  M, N = x.shape

  grid = lambda meta: (triton.cdiv(M, meta['BLOCK_M']),triton.cdiv(N, meta['BLOCK_N']))

  matrix_transpose_kernel[grid](
      x,
      out,
      M, N,
      BLOCK_M=32,
      BLOCK_N=32
  )

  return out

x2d = torch.arange(100000, dtype=torch.float32).view(1000,100).to(device)
print(x2d)
print(matrix_transpose(x2d))

tensor([[0.0000e+00, 1.0000e+00, 2.0000e+00,  ..., 9.7000e+01, 9.8000e+01,
         9.9000e+01],
        [1.0000e+02, 1.0100e+02, 1.0200e+02,  ..., 1.9700e+02, 1.9800e+02,
         1.9900e+02],
        [2.0000e+02, 2.0100e+02, 2.0200e+02,  ..., 2.9700e+02, 2.9800e+02,
         2.9900e+02],
        ...,
        [9.9700e+04, 9.9701e+04, 9.9702e+04,  ..., 9.9797e+04, 9.9798e+04,
         9.9799e+04],
        [9.9800e+04, 9.9801e+04, 9.9802e+04,  ..., 9.9897e+04, 9.9898e+04,
         9.9899e+04],
        [9.9900e+04, 9.9901e+04, 9.9902e+04,  ..., 9.9997e+04, 9.9998e+04,
         9.9999e+04]], device='cuda:0')
tensor([[0.0000e+00, 1.0000e+02, 2.0000e+02,  ..., 9.9700e+04, 9.9800e+04,
         9.9900e+04],
        [1.0000e+00, 1.0100e+02, 2.0100e+02,  ..., 9.9701e+04, 9.9801e+04,
         9.9901e+04],
        [2.0000e+00, 1.0200e+02, 2.0200e+02,  ..., 9.9702e+04, 9.9802e+04,
         9.9902e+04],
        ...,
        [9.7000e+01, 1.9700e+02, 2.9700e+02,  ..., 9.9797e+04, 9.9897e+04,
        

In [ ]:
@triton.jit
def tiled_matmul_kernel(
    x_ptr,
    y_ptr,
    out_ptr,
    M, K, N,
    stride_xm, stride_xk,
    stride_yk, stride_yn,
    stride_outm, stride_outn,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr,
    BLOCK_K : tl.constexpr
):

  # set out important assetions before hand for all the shapes
  pid_m = tl.program_id(0)
  pid_n = tl.program_id(1)

  offsets_m = pid_m*BLOCK_M + tl.arange(0, BLOCK_M)
  offsets_n = pid_n*BLOCK_N + tl.arange(0, BLOCK_N)
  offsets_k = tl.arange(0, BLOCK_K)

  offset_matrix_x = offsets_m[:,None]*stride_xm + offsets_k[None,:]*stride_xk
  offset_matrix_y = offsets_k[:,None]*stride_yk + offsets_n[None,:]*stride_yn
  offset_matrix_out = offsets_m[:,None]*stride_outm + offsets_n[None,:]*stride_outn

  x_ptr = x_ptr + offset_matrix_x
  y_ptr = y_ptr + offset_matrix_y
  out_ptr = out_ptr + offset_matrix_out

  out = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

  for k in range(0, K, BLOCK_K):
    mask_x = (offsets_m[:,None] < M) & (offsets_k[None,:]<(K-k))
    mask_y = (offsets_k[:,None] < (K-k)) & (offsets_n[None,:]<N)

    x = tl.load(x_ptr, mask=mask_x, other=0.0)
    y = tl.load(y_ptr, mask=mask_y, other=0.0)

    out += tl.dot(x,y)

    x_ptr += BLOCK_K*stride_xk
    y_ptr += BLOCK_K*stride_yk

  out_mask = (offsets_m[:,None] < M) & (offsets_n[None,:]<N)
  tl.store(out_ptr, out, mask=out_mask)

def tiled_matmul(x : torch.Tensor, y : torch.Tensor):
  M,K = x.shape
  Kt, N = y.shape
  assert K==Kt
  assert x.is_contiguous()
  assert y.is_contiguous()
  assert x.is_cuda
  assert y.is_cuda

  out = torch.empty((M,N), dtype=torch.float32).to(device).contiguous()

  grid = lambda meta : (triton.cdiv(M,meta['BLOCK_M']),triton.cdiv(N,meta['BLOCK_N']))

  tiled_matmul_kernel[grid](
      x,
      y,
      out,
      M,K,N,
      x.stride(0), x.stride(1),
      y.stride(0), y.stride(1),
      out.stride(0), out.stride(1),
      BLOCK_M=64,
      BLOCK_N=64,
      BLOCK_K=32
  )
  return out

x2d = torch.arange(12, dtype=torch.float32).to(device).view(3,4)
y2d = torch.arange(20, dtype=torch.float32).to(device).view(4,5)

tiled_matmul(x2d, y2d)

In [ ]:
@triton.jit
def online_softmax_kernel(
    x_ptr,
    y_ptr,
    M, N,
    stride_xm, stride_xn,
    stride_ym, stride_yn,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)

  offsets_x = pid*stride_xm + tl.arange(0,BLOCK_SIZE)*stride_xn()
  prev_max = -float('inf')
  denominator = 0.0

  for n in range(0, N, BLOCK_SIZE):
    mask = tl.arange(0,BLOCK_SIZE) < (N - n)
    block_x = tl.load(x_ptr+offsets_x, mask=mask, other = -float('inf'))

    block_max = tl.max(block_x, axis = 0)
    current_max = tl.maximum(block_max, prev_max)

    scaling_factor = tl.exp(prev_max-current_max)

    exponents = tl.exp(block_x-current_max)

    denominator = denominator*scaling_factor + tl.sum(exponents, axis= 0)

    prev_max = current_max

    offsets_x = offsets_x + BLOCK_SIZE*stride_xn

  offsets_x = pid*stride_xm + tl.arange(0,BLOCK_SIZE)*stride_xn()
  offsets_y = pid*stride_ym + tl.arange(0,BLOCK_SIZE)*stride_yn()
  inv_denom = 1.0 / denominator
  for n in range(0, N, BLOCK_SIZE):
    mask = tl.arange(0,BLOCK_SIZE) < (N - n)
    block_x = tl.load(x_ptr+offsets_x, mask=mask)
    normals = tl.exp(block_x-prev_max)*inv_denom
    tl.store(y_ptr+offsets_y,normals,mask=mask)
    offsets_x = offsets_x +BLOCK_SIZE*stride_xn
    offsets_y = offsets_y + BLOCK_SIZE*stride_yn

def online_softmax(x : torch.Tensor):
  assert x.is_cuda
  assert x.is_contiguous()

  y = torch.empty_like(x)
  M, N = x.shape

  grid = lambda meta : (M,)

  online_softmax_kernel[grid](
      x,
      y,
      M,N,
      x.stride(0), x.stride(1),
      y.stride(0), y.stride(1),
      BLOCK_SIZE = 1024
  )
  return y

In [ ]:
import triton
import triton.language as tl
import math
import torch

@triton.jit
def flash_attention_kernel(
    Q_ptr, # M D
    K_ptr, # N D
    V_ptr, # N D
    O_ptr, # M D
    LSE_ptr,
    M, N,
    stride_qm, stride_qd, stride_qb, stride_qh,
    stride_kn, stride_kd, stride_kb, stride_kh,
    stride_vn, stride_vd, stride_vb, stride_vh,
    stride_om, stride_od, stride_ob, stride_oh,
    stride_lseb, stride_lseh, stride_lsem,
    div_factor,
    HEADS_PER_BATCH : tl.constexpr,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr,
    WHOLE_D : tl.constexpr
):
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  Q_ptr = Q_ptr + current_batch * stride_qb + current_head * stride_qh
  K_ptr = K_ptr + current_batch * stride_kb + current_head * stride_kh
  V_ptr = V_ptr + current_batch * stride_vb + current_head * stride_vh
  O_ptr = O_ptr + current_batch * stride_ob + current_head * stride_oh
  LSE_ptr = LSE_ptr + current_batch * stride_lseb + current_head * stride_lseh

  offsets_m = BLOCK_M * pid + tl.arange(0 , BLOCK_M)
  offsets_n = tl.arange(0 , BLOCK_N)
  offsets_d = tl.arange(0, WHOLE_D)

  mask_q = offsets_m < M

  offsets_matrix_Q = offsets_m[:,None] * stride_qm + offsets_d[None,:] * stride_qd
  q_block = tl.load(Q_ptr + offsets_matrix_Q, mask_q[:, None], other = 0.0)

  offsets_matrix_K = offsets_n[None,:] * stride_kn + offsets_d[:,None] * stride_kd
  offsets_matrix_V = offsets_n[:,None] * stride_vn + offsets_d[None,:] * stride_vd

  prev_max_per_row = tl.zeros([BLOCK_M], dtype=tl.float32) - float('inf')
  denominator_per_row = tl.zeros([BLOCK_M], dtype=tl.float32)

  accumulator = tl.zeros([BLOCK_M, WHOLE_D], dtype=tl.float32)

  end_n = tl.minimum((pid+1)*BLOCK_M, N)

  for n in range(0, end_n, BLOCK_N):
    mask_kv = offsets_n < (N-n)
    causal_mask = (offsets_m[:, None]>=(n + offsets_n)[None, :]) & mask_kv[None, :]

    k_block = tl.load(K_ptr + offsets_matrix_K, mask_kv[None, :], other = 0.0 )
    v_block = tl.load(V_ptr + offsets_matrix_V, mask_kv[:, None], other = 0.0)

    qk_block = tl.dot(q_block,k_block)*div_factor
    qk_block = tl.where(causal_mask, qk_block, -float('inf'))

    temp_max_per_row = tl.max(qk_block,axis=1)

    current_max_per_row = tl.maximum(temp_max_per_row, prev_max_per_row)

    score = tl.exp(qk_block-current_max_per_row[:, None])

    alpha_per_row = tl.exp(prev_max_per_row-current_max_per_row)

    denominator_per_row = denominator_per_row*alpha_per_row + tl.sum(score, axis = -1)

    p = tl.dot(score.to(v_block.dtype), v_block)

    accumulator = accumulator*alpha_per_row[:, None] + p

    prev_max_per_row = current_max_per_row

    offsets_matrix_K = offsets_matrix_K + BLOCK_N*stride_kn
    offsets_matrix_V = offsets_matrix_V + BLOCK_N*stride_vn

  # Prevent division by zero and log of zero/negative values
  denominator_per_row_safe = tl.where(denominator_per_row > 0, denominator_per_row, 1e-6)
  accumulator = accumulator / denominator_per_row_safe[:, None]

  lse = prev_max_per_row + tl.log(denominator_per_row_safe)
  tl.store(LSE_ptr+offsets_m,lse,mask_q)

  offsets_matrix_O = offsets_m[:, None]* stride_om + offsets_d[None, :] * stride_od
  tl.store(O_ptr + offsets_matrix_O, accumulator.to(q_block.dtype), mask_q[: , None])


def flash_attention(
    Q : torch.Tensor,
    K : torch.Tensor,
    V : torch.Tensor,
    sm_scale : float = None
):
  assert Q.is_cuda and Q.is_contiguous()
  assert K.is_cuda and K.is_contiguous()
  assert V.is_cuda and V.is_contiguous()

  assert Q.shape[-1] == K.shape[-1]
  assert K.shape == V.shape

  B, H, T, C = Q.shape
  _, _, T2, _ = K.shape


  grid = lambda meta : (triton.cdiv(T, meta['BLOCK_M']),B*H)

  if sm_scale is None:
        sm_scale = 1.0 / (D ** 0.5)

  O = torch.empty_like(Q)
  LSE = torch.empty((B, H, T), dtype=torch.float32, device=Q.device)

  flash_attention_kernel[grid](
      Q,
      K,
      V,
      O,
      LSE,
      T, T2,
      Q.stride(2), Q.stride(3), Q.stride(0), Q.stride(1),
      K.stride(2), K.stride(3), K.stride(0), K.stride(1),
      V.stride(2), V.stride(3), V.stride(0), V.stride(1),
      O.stride(2), O.stride(3), O.stride(0), O.stride(1),
      LSE.stride(0), LSE.stride(1), LSE.stride(2),
      sm_scale,
      HEADS_PER_BATCH = H,
      BLOCK_M = 32,
      BLOCK_N = 32,
      WHOLE_D = C
    )

  return O, LSE

In [ ]:
# The backward pass

@triton.jit
def flash_attention_backward_pre_kernel(
    O_ptr,
    dO_ptr,
    D_ptr,
    M,
    stride_ob, stride_oh, stride_om, stride_od,
    stride_dob, stride_doh, stride_dom, stride_dod,
    stride_db, stride_dh, stride_dm,
    HEAD_DATA: tl.constexpr,
    BLOCK_M: tl.constexpr,
    HEADS_PER_BATCH: tl.constexpr
):
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  O_ptr = O_ptr + current_batch*stride_ob + current_head*stride_oh
  dO_ptr = dO_ptr + current_batch*stride_dob + current_head*stride_doh
  D_ptr = D_ptr + current_batch*stride_db + current_head*stride_dh

  offsets_m = pid*BLOCK_M + tl.arange(0,BLOCK_M)
  offsets_d = tl.arange(0,HEAD_DATA)

  offsets_matrix_o = offsets_m[:,None]*stride_om + offsets_d[None,:]*stride_od
  offsets_matrix_do = offsets_m[:,None]*stride_dom + offsets_d[None,:]*stride_dod

  mask_m = offsets_m < M

  O = tl.load(O_ptr+offsets_matrix_o, mask_m[:,None], other=0.0)
  dO = tl.load(dO_ptr+offsets_matrix_do, mask_m[:, None], other=0.0)

  D = tl.sum(O*dO, axis=1)

  tl.store(D_ptr + offsets_m*stride_dm, D, mask_m)


@triton.jit
def flash_attention_backward_kernel(
    Q_ptr, K_ptr, V_ptr, LSE_ptr, Delta_ptr, dO_ptr,
    dQ_ptr, dK_ptr, dV_ptr,
    M, N,
    sm_scale,
    stride_qb, stride_qh, stride_qm, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_dob, stride_doh, stride_dom, stride_dod,
    stride_dqb, stride_dqh, stride_dqm, stride_dqd,
    stride_dkb, stride_dkh, stride_dkn, stride_dkd,
    stride_dvb, stride_dvh, stride_dvn, stride_dvd,
    stride_lseb, stride_lseh, stride_lsem,
    stride_deltab, stride_deltah, stride_deltam,
    HEADS_PER_BATCH: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    HEAD_DIM: tl.constexpr,
):
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  Q_ptr = Q_ptr + current_batch*stride_qb + current_head*stride_qh
  K_ptr = K_ptr + current_batch*stride_kb + current_head*stride_kh
  V_ptr = V_ptr + current_batch*stride_vb + current_head*stride_vh
  dO_ptr = dO_ptr + current_batch*stride_dob + current_head*stride_doh
  dQ_ptr = dQ_ptr + current_batch*stride_dqb + current_head*stride_dqh
  dK_ptr = dK_ptr + current_batch*stride_dkb + current_head*stride_dkh
  dV_ptr = dV_ptr + current_batch*stride_dvb + current_head*stride_dvh
  LSE_ptr = LSE_ptr + current_batch*stride_lseb + current_head*stride_lseh
  Delta_ptr = Delta_ptr + current_batch*stride_deltab + current_head*stride_deltah #causal

  offsets_m = tl.arange(0,BLOCK_M)
  offsets_n = pid*BLOCK_N + tl.arange(0,BLOCK_N)
  offsets_d = tl.arange(0, HEAD_DIM)

  offsets_matrix_q = offsets_m[:,None]*stride_qm + offsets_d[None,:]*stride_qd
  offsets_matrix_k = offsets_n[:, None]*stride_kn + offsets_d[None,:]*stride_kd
  offsets_matrix_v = offsets_n[:,None]*stride_vn + offsets_d[None,:]*stride_vd
  offsets_matrix_do = offsets_m[:,None]*stride_dom + offsets_d[None,:]*stride_dod
  offsets_matrix_dq = offsets_m[:,None]*stride_dqm + offsets_d[None,:]*stride_dqd
  offsets_matrix_dk = offsets_n[:,None]*stride_dkn + offsets_d[None,:]*stride_dkd
  offsets_matrix_dv = offsets_n[:,None]*stride_dvn + offsets_d[None,:]*stride_dvd
  offsets_matrix_lse = offsets_m*stride_lsem
  offsets_matrix_delta = offsets_m*stride_deltam

  mask_m = offsets_m < M
  mask_n = offsets_n < N

  k_block = tl.load(K_ptr + offsets_matrix_k, mask_n[:,None], other=0.0) # bN * D
  v_block = tl.load(V_ptr + offsets_matrix_v, mask_n[:,None], other=0.0) # bN * D
  dk_block = tl.zeros([BLOCK_N, HEAD_DIM], dtype=tl.float32)
  dv_block = tl.zeros([BLOCK_N, HEAD_DIM], dtype=tl.float32)

  for m in range(0, M, BLOCK_M):
    mask_m = offsets_m < (M-m)
    q_block = tl.load(Q_ptr + offsets_matrix_q, mask_m[:, None], other=0.0)
    do_block = tl.load(dO_ptr + offsets_matrix_do, mask_m[:, None], other=0.0)
    LSE_block = tl.load(LSE_ptr + offsets_matrix_lse, mask_m, other=-10000.0)
    Delta_block = tl.load(Delta_ptr + offsets_matrix_delta, mask_m, other=0.0)

    qk_block = tl.dot(q_block, tl.trans(k_block))*sm_scale

    # Prevent NaN values during exponentiation by clamping extremely low LSE values
    lse_clamped = tl.where(LSE_block > -1000.0, LSE_block, -1000.0)
    p_block = tl.exp(qk_block - lse_clamped[:, None])

    causal_mask = ((offsets_m + m)[:, None] >= (offsets_n)[None, :]) & mask_m[:, None] & mask_n[None, :]
    p_block = tl.where(causal_mask, p_block, 0.0)
    p_block = p_block.to(do_block.dtype)

    dv_block += tl.dot(tl.trans(p_block), do_block)

    dp_block = tl.dot(do_block, tl.trans(v_block))

    dqk_block = p_block * (dp_block - Delta_block[:, None]) * sm_scale
    dqk_block = tl.where(causal_mask, dqk_block, 0.0)
    dqk_block = dqk_block.to(k_block.dtype)

    dk_block += tl.dot(tl.trans(dqk_block) , q_block)

    sub_q_block = tl.dot(dqk_block , k_block)

    tl.atomic_add(dQ_ptr + offsets_matrix_dq, sub_q_block, mask_m[:, None])

    offsets_matrix_q += BLOCK_M*stride_qm
    offsets_matrix_dq += BLOCK_M*stride_dqm
    offsets_matrix_do += BLOCK_M*stride_dom
    offsets_matrix_lse += BLOCK_M*stride_lsem
    offsets_matrix_delta += BLOCK_M*stride_deltam

  tl.store(dK_ptr + offsets_matrix_dk, dk_block.to(k_block.dtype), mask=mask_n[:, None])
  tl.store(dV_ptr + offsets_matrix_dv, dv_block.to(v_block.dtype), mask=mask_n[:, None])


def flash_attention_backward(
    dO: torch.Tensor,
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    O: torch.Tensor,
    LSE: torch.Tensor,
    sm_scale: float = 0
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

    # Enforce contiguous layout
    dO = dO.contiguous()
    Q = Q.contiguous()
    K = K.contiguous()
    V = V.contiguous()
    O = O.contiguous()
    LSE = LSE.contiguous()

    B, H, M, D = Q.shape
    _, _, N, _ = K.shape

    if sm_scale is None:
        sm_scale = 1.0 / (D ** 0.5)

    Delta = torch.empty((B, H, M), device=Q.device, dtype=torch.float32)

    dQ = torch.zeros_like(Q, dtype=torch.float32)
    dK = torch.empty_like(K)
    dV = torch.empty_like(V)

    BLOCK_M = 32
    BLOCK_N = 16

    grid_pre = (triton.cdiv(M, BLOCK_M), B * H)

    flash_attention_backward_pre_kernel[grid_pre](
        O_ptr=O,
        dO_ptr=dO,
        D_ptr=Delta,
        M=M,
        stride_ob=O.stride(0), stride_oh=O.stride(1), stride_om=O.stride(2), stride_od=O.stride(3),
        stride_dob=dO.stride(0), stride_doh=dO.stride(1), stride_dom=dO.stride(2), stride_dod=dO.stride(3),
        stride_db=Delta.stride(0), stride_dh=Delta.stride(1), stride_dm=Delta.stride(2),
        HEAD_DATA=D,
        BLOCK_M=BLOCK_M,
        HEADS_PER_BATCH=H,
    )

    grid_bwd = (triton.cdiv(N, BLOCK_N), B * H)

    flash_attention_backward_kernel[grid_bwd](
        Q_ptr=Q, K_ptr=K, V_ptr=V,
        LSE_ptr=LSE, Delta_ptr=Delta, dO_ptr=dO,
        dQ_ptr=dQ, dK_ptr=dK, dV_ptr=dV,
        M=M, N=N,
        sm_scale=sm_scale,
        stride_qb=Q.stride(0), stride_qh=Q.stride(1), stride_qm=Q.stride(2), stride_qd=Q.stride(3),
        stride_kb=K.stride(0), stride_kh=K.stride(1), stride_kn=K.stride(2), stride_kd=K.stride(3),
        stride_vb=V.stride(0), stride_vh=V.stride(1), stride_vn=V.stride(2), stride_vd=V.stride(3),
        stride_dob=dO.stride(0), stride_doh=dO.stride(1), stride_dom=dO.stride(2), stride_dod=dO.stride(3),
        stride_dqb=dQ.stride(0), stride_dqh=dQ.stride(1), stride_dqm=dQ.stride(2), stride_dqd=dQ.stride(3),
        stride_dkb=dK.stride(0), stride_dkh=dK.stride(1), stride_dkn=dK.stride(2), stride_dkd=dK.stride(3),
        stride_dvb=dV.stride(0), stride_dvh=dV.stride(1), stride_dvn=dV.stride(2), stride_dvd=dV.stride(3),
        stride_lseb=LSE.stride(0), stride_lseh=LSE.stride(1), stride_lsem=LSE.stride(2),
        stride_deltab=Delta.stride(0), stride_deltah=Delta.stride(1), stride_deltam=Delta.stride(2),
        HEADS_PER_BATCH=H,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        HEAD_DIM=D,
    )

    return dQ, dK, dV

In [ ]:
class custom_flash_attention(torch.autograd.Function):
  @staticmethod
  def forward(ctx, Q, K, V, sm_scale=None):
    #print('Forward_debug')
    if sm_scale is None:
      sm_scale = 1.0 / (Q.shape[-1] ** 0.5)
    O, LSE = flash_attention(Q, K, V, sm_scale)
    ctx.save_for_backward(Q, K, V, O, LSE)
    ctx.sm_scale = sm_scale
    return O

  @staticmethod
  def backward(ctx, dO):
    #print('backward_debug')
    Q, K, V, O, LSE = ctx.saved_tensors
    dQ, dK, dV = flash_attention_backward(
            dO=dO,
            Q=Q,
            K=K,
            V=V,
            O=O,
            LSE=LSE,
            sm_scale = ctx.sm_scale
        )
    return dQ, dK, dV, None